# Lab 3 - Datasets and Tokenizers

**Day 1 - Pre-trained Models** | Estimated time: **40 minutes** | Runs on CPU

Every LLM sees text as a list of **token IDs**, never as characters or words. In this lab you will:

1. Explore a real dataset with the Hugging Face `datasets` library
2. Watch a tokenizer turn text into IDs, special tokens and attention masks
3. Compare five tokenizers and discover the **"language tax"** on Hindi and Kannada
4. Tokenize a whole dataset and choose a sensible `max_length`
5. Estimate what an LLM job would **cost**, and cut it with batching and caching strategies
6. *(Stretch)* Train your own tokenizer

> **How to run:** click inside a code cell and press **Shift+Enter**. Run the cells in order, top to bottom.

In [ ]:
# Setup: quieter logs and UTF-8 output on Windows
import os, sys
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
# Load optional settings (e.g. HF_TOKEN) from the repository's .env file
from pathlib import Path
from dotenv import load_dotenv
for folder in [Path.cwd(), *Path.cwd().parents][:4]:
    if (folder / ".env").exists():
        load_dotenv(folder / ".env")
        break
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

import datasets, transformers, tiktoken
print("datasets    ", datasets.__version__)
print("transformers", transformers.__version__)
print("tiktoken    ", tiktoken.__version__)

def show_tokens(tokens, limit=40):
    """Print tokens with visible boundaries: |tok|tok|tok|"""
    more = f" ... (+{len(tokens) - limit} more)" if len(tokens) > limit else ""
    return "|" + "|".join(tokens[:limit]) + "|" + more

---
## Part 1 - Explore a dataset (10 min)

We use **IMDB**: 50,000 movie reviews labelled positive or negative, plus 50,000 unlabelled ones.
`load_dataset` downloads it from the Hugging Face Hub once and caches it locally.

In [ ]:
from datasets import load_dataset

ds = load_dataset("stanfordnlp/imdb")
ds

A `DatasetDict` holds one `Dataset` per **split**. Each `Dataset` is a table backed by Apache Arrow,
so it is memory-mapped from disk rather than loaded into RAM.

Look at the **schema**: `label` is a `ClassLabel`, which stores integers but knows their names.

In [ ]:
train = ds["train"]
print(train.features)

label_feature = train.features["label"]
print("Label names:", label_feature.names)
print("int2str(1) ->", label_feature.int2str(1))

In [ ]:
# One row -> a dict.  A slice -> a dict of lists.
first = train[0]
print("Label:", label_feature.int2str(first["label"]))
print(first["text"][:400], "...")

print("\nSlice shape:", {k: len(v) for k, v in train[:3].items()})

In [ ]:
# Is the dataset balanced?
from collections import Counter
for split in ["train", "test", "unsupervised"]:
    counts = Counter(ds[split]["label"])
    print(f"{split:<13}", dict(sorted(counts.items())))

**`map()`** adds or transforms columns. **`filter()`** keeps the rows that match a condition.
Both return a *new* dataset and cache the result on disk.

In [ ]:
train = train.map(lambda ex: {"n_words": len(ex["text"].split())})

lengths = sorted(train["n_words"])
n = len(lengths)
print(f"Words per review: min={lengths[0]}  median={lengths[n//2]}  "
      f"p95={lengths[int(n*0.95)]}  max={lengths[-1]}")

short = train.filter(lambda ex: ex["n_words"] < 50)
print(f"{len(short)} of {len(train)} reviews are under 50 words")
print("Example:", short[0]["text"][:200])

In [ ]:
# Build a small, balanced working set: shuffle -> select -> stratified split
small = train.shuffle(seed=42).select(range(2000))
splits = small.train_test_split(test_size=0.2, seed=42, stratify_by_column="label")
splits

In [ ]:
# Hand off to pandas whenever you want classic data analysis
df = small.to_pandas()
df["label_name"] = df["label"].map(label_feature.int2str)
df.groupby("label_name")["n_words"].describe().round(1)

In [ ]:
# Save and reload
os.makedirs("outputs", exist_ok=True)
splits.save_to_disk("outputs/imdb_small")
splits["train"].to_csv("outputs/imdb_small_train.csv")

from datasets import load_from_disk
print(load_from_disk("outputs/imdb_small"))

In [ ]:
# Streaming: iterate over a dataset without downloading all of it
stream = load_dataset("stanfordnlp/imdb", split="test", streaming=True)
for i, ex in enumerate(stream.take(3)):
    print(f"[{i}] {label_feature.int2str(ex['label'])}: {ex['text'][:90]}...")

> **Try it (2 min)**
> 1. How many reviews in `train` are longer than 1,000 words? Use `filter()`.
> 2. Is the average review length different for `pos` and `neg`? Use the pandas DataFrame `df`.

In [ ]:
# Your code here

---
## Part 2 - How a tokenizer works (8 min)

We start with the **BERT** tokenizer, which uses the **WordPiece** algorithm.
`AutoTokenizer` downloads the vocabulary that matches a model on the Hub.

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
print("Vocabulary size:", f"{tok.vocab_size:,}")
print("Max length     :", tok.model_max_length)
print("Special tokens :", tok.all_special_tokens)

In [ ]:
text = "Tokenization isn't magic: transformers read sub-words!"

tokens = tok.tokenize(text)
ids = tok.convert_tokens_to_ids(tokens)
print("Tokens:", show_tokens(tokens))
print("IDs   :", ids)

`##` means *"this piece continues the previous word"*. Common words get a single token.
Rarer words are split into known sub-words, so the tokenizer **never needs an out-of-vocabulary word**.

Calling the tokenizer directly does everything a model needs, including the special tokens:

In [ ]:
enc = tok(text)
print("input_ids     :", enc["input_ids"])
print("tokens        :", show_tokens(tok.convert_ids_to_tokens(enc["input_ids"])))
print("attention_mask:", enc["attention_mask"])
print("decoded       :", tok.decode(enc["input_ids"], skip_special_tokens=True))

In [ ]:
for word in ["cat", "unbelievably", "Mysuru", "COVID-19", "😀"]:
    print(f"{word:<14} -> {show_tokens(tok.tokenize(word))}")

The emoji became `[UNK]` (unknown) because BERT's 30k vocabulary has no piece for it. Keep that in mind for Part 3.

### Batches: padding, truncation and the attention mask

In [ ]:
batch = ["Short one.",
         "A slightly longer sentence than the first.",
         "This third sentence is deliberately much longer than the other two so it gets cut."]

enc = tok(batch, padding=True, truncation=True, max_length=12)
for i, ids in enumerate(enc["input_ids"]):
    print(f"[{i}] {show_tokens(tok.convert_ids_to_tokens(ids))}")
    print(f"    mask {enc['attention_mask'][i]}")

* `[PAD]` makes every sequence in the batch the same length (models need rectangular tensors).
* `attention_mask = 0` tells the model to **ignore** the padding.
* Sentence `[2]` was **truncated** at 12 tokens, so the end of that text is gone.

In [ ]:
tensors = tok(batch, padding=True, return_tensors="pt")
print("input_ids tensor shape:", tuple(tensors["input_ids"].shape), "= (batch, sequence_length)")

# Offsets map every token back to the characters it came from
enc = tok("Mysuru Palace glows at night", return_offsets_mapping=True)
for t, (s, e) in zip(tok.convert_ids_to_tokens(enc["input_ids"]), enc["offset_mapping"]):
    print(f"{t:<10} chars {s:>2}-{e:<2}")

---
## Part 3 - Compare tokenizers and the "language tax" (8 min)

| Tokenizer | Algorithm | Used by |
|---|---|---|
| BERT | WordPiece | encoder models (classification, NER) |
| GPT-2 | byte-level BPE | the original GPT family |
| XLM-RoBERTa | SentencePiece (Unigram) | multilingual encoder, 100 languages |
| `cl100k_base` | BPE (tiktoken) | GPT-4, GPT-3.5-turbo |
| `o200k_base` | BPE (tiktoken) | GPT-4o-family models |

LLM APIs **bill per token**, and context windows are measured in tokens. Let's count.

In [ ]:
SAMPLES = {
    "English": "The quarterly report shows revenue grew by 12% year over year.",
    "Code":    "def add(a, b):\n    return a + b",
    "Numbers": "Order #847261 shipped on 2026-11-15 for Rs. 1,24,999.00",
    "JSON":    '{"customer_id": 1042, "status": "active", "tier": "gold"}',
    "Hindi":   "तिमाही रिपोर्ट से पता चलता है कि राजस्व में 12% की वृद्धि हुई।",
    "Kannada": "ತ್ರೈಮಾಸಿಕ ವರದಿಯು ಆದಾಯ 12% ಹೆಚ್ಚಾಗಿದೆ ಎಂದು ತೋರಿಸುತ್ತದೆ.",
    "Emoji":   "Great service 👍🔥 will order again 😀",
}

hf_toks = {
    "BERT":  AutoTokenizer.from_pretrained("google-bert/bert-base-uncased"),
    "GPT-2": AutoTokenizer.from_pretrained("openai-community/gpt2"),
    "XLM-R": AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-base"),
}
tk = {"cl100k": tiktoken.get_encoding("cl100k_base"),
      "o200k":  tiktoken.get_encoding("o200k_base")}

def count(name, text):
    if name in hf_toks:
        return len(hf_toks[name].encode(text, add_special_tokens=False))
    return len(tk[name].encode(text))

def pieces(name, text):
    if name in hf_toks:
        return hf_toks[name].tokenize(text)
    enc = tk[name]
    return [enc.decode_single_token_bytes(i).decode("utf-8", errors="replace") for i in enc.encode(text)]

import pandas as pd
names = list(hf_toks) + list(tk)
table = pd.DataFrame({n: {lbl: count(n, t) for lbl, t in SAMPLES.items()} for n in names})
table.insert(0, "chars", [len(t) for t in SAMPLES.values()])
table

In [ ]:
for label in ["English", "Hindi"]:
    for n in ["GPT-2", "o200k"]:
        p = pieces(n, SAMPLES[label])
        print(f"{label} | {n} | {len(p)} tokens\n   {show_tokens(p, 30)}\n")

In [ ]:
# The language tax: same meaning, how many times the English token count?
for n in ["GPT-2", "cl100k", "o200k"]:
    base = count(n, SAMPLES["English"])
    print(f"{n:<7} Hindi x{count(n, SAMPLES['Hindi'])/base:.1f}   "
          f"Kannada x{count(n, SAMPLES['Kannada'])/base:.1f}")

**What to notice**

* GPT-2's merges were learned mostly from English web text, so Hindi and Kannada fall apart into **byte fragments**.
* `o200k_base` has a much larger, more multilingual vocabulary, so the tax shrinks a lot.
* A high token count means **higher cost, higher latency, and less room** in the context window.

> **Try it (2 min):** add a sentence in your own language, or a SQL query, to `SAMPLES` and re-run the table.
> Also compare `"hello"`, `" hello"`, `"Hello"` and `"HELLO"` with `pieces("o200k", ...)`. Spaces and casing matter!

In [ ]:
# Your code here
print(pieces("o200k", "HELLO"), pieces("o200k", " hello"))

---
## Part 4 - Tokenize a whole dataset and choose `max_length` (6 min)

We now tokenize 5,000 reviews with the DistilBERT tokenizer, the same model you fine-tune in Lab 2.

In [ ]:
import time
from transformers import DataCollatorWithPadding

tok = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")
sample = ds["train"].shuffle(seed=0).select(range(5000))

t0 = time.time()
tokenized = sample.map(lambda b: tok(b["text"]), batched=True, batch_size=1000)
print(f"Tokenized {len(sample)} reviews in {time.time()-t0:.1f}s with batched=True")
print("Columns:", tokenized.column_names)

In [ ]:
lengths = sorted(len(ids) for ids in tokenized["input_ids"])
p = lambda q: lengths[min(len(lengths)-1, int(len(lengths)*q))]
print(f"Tokens per review: min={lengths[0]} median={p(.5)} p90={p(.9)} p95={p(.95)} max={lengths[-1]}")
print(f"DistilBERT can read at most {tok.model_max_length} tokens\n")

total = sum(lengths)
rows = []
for m in [64, 128, 256, 512]:
    rows.append({"max_length": m,
                 "reviews truncated": f"{sum(n > m for n in lengths)/len(lengths):.1%}",
                 "tokens discarded": f"{sum(n-m for n in lengths if n > m)/total:.1%}"})
pd.DataFrame(rows)

Longer `max_length` keeps more of each review but costs memory and time
(attention cost grows roughly with the **square** of the sequence length).

### How much compute is wasted on padding?

In [ ]:
MAXLEN, BATCH = 512, 16
trunc = [min(n, MAXLEN) for n in (len(ids) for ids in tokenized["input_ids"])]
real = sum(trunc)

def padded_total(seq):
    return sum(max(seq[i:i+BATCH]) * len(seq[i:i+BATCH]) for i in range(0, len(seq), BATCH))

for name, total_tokens in [("Static: pad everything to 512", len(trunc) * MAXLEN),
                           ("Dynamic: pad to longest in batch", padded_total(trunc)),
                           ("Dynamic + length-grouped batches", padded_total(sorted(trunc)))]:
    print(f"{name:<34} {total_tokens:>10,} tokens  {(total_tokens-real)/total_tokens:6.1%} padding")

IMDB has a long tail of very long reviews, so almost every random batch contains one, and dynamic padding alone barely helps.
**Grouping similar lengths** removes nearly all the waste. In the Hugging Face Trainer that is
`TrainingArguments(train_sampling_strategy="group_by_length")`.

`DataCollatorWithPadding` (used by the Trainer in Lab 2) does the dynamic padding for you:

In [ ]:
collator = DataCollatorWithPadding(tok, return_tensors="pt")
features = [{"input_ids": ids[:MAXLEN]} for ids in tokenized["input_ids"][:4]]
batch = collator(features)
print("Lengths in batch:", [len(f["input_ids"]) for f in features])
print("Padded shape    :", tuple(batch["input_ids"].shape))

---
## Part 5 - Token budgeting: what would this cost on an LLM API? (8 min)

**Scenario:** your team wants an LLM to label the sentiment of all **25,000** IMDB test reviews.
Before sending a single request, you count tokens with `tiktoken` and compare strategies.

> The prices below are **example values** for a small "mini"-class model.
> Always plug in the current figures from your provider's pricing page.

In [ ]:
PRICE = {"input": 0.40, "cached_input": 0.10, "output": 1.60}   # USD per 1M tokens (EXAMPLE values)
BATCH_API_DISCOUNT = 0.50     # async batch jobs are often ~50% cheaper
CACHE_MIN_PREFIX = 1024       # prompt caching usually needs an identical prefix of >= 1024 tokens
REVIEWS_PER_REQUEST = 20
TRUNCATE_TO = 300

enc = tiktoken.get_encoding("o200k_base")
test = ds["test"]
N = len(test)
measure = test.shuffle(seed=1).select(range(2500))      # measure a sample, scale up
scale = N / len(measure)

INSTRUCTIONS = (
    "You are a sentiment analysis service for a film studio.\n"
    "Classify each movie review as positive or negative.\n"
    "Rules:\n- Judge the reviewer's overall opinion of the film, not the plot's mood.\n"
    "- Sarcasm counts as the opinion it expresses.\n"
    "- Mixed reviews: choose the side the reviewer ends on.\n")

# 10 few-shot examples make the prompt better, and much longer
shots = []
for ex in ds["train"].shuffle(seed=7).select(range(10)):
    trimmed = enc.decode(enc.encode(ex["text"])[:110])
    shots.append(f"Review: {trimmed}\nLabel: {'positive' if ex['label'] else 'negative'}")
SYSTEM = INSTRUCTIONS + "\nExamples:\n\n" + "\n\n".join(shots)

t_system = len(enc.encode(SYSTEM))
review_tokens = [len(enc.encode(t)) for t in measure["text"]]
print(f"System prompt (instructions + 10 examples): {t_system} tokens "
      f"-> {'eligible' if t_system >= CACHE_MIN_PREFIX else 'too short'} for prompt caching")
print(f"Average review: {sum(review_tokens)/len(review_tokens):.0f} tokens (max {max(review_tokens):,})")

In [ ]:
def cost(n_in, n_out, n_cached=0, batch_api=False):
    usd = ((n_in - n_cached) * PRICE["input"] + n_cached * PRICE["cached_input"]
           + n_out * PRICE["output"]) / 1_000_000
    return usd * (1 - BATCH_API_DISCOUNT) if batch_api else usd

OVERHEAD, VERBOSE_OUT, LABEL_OUT = 8, 60, 2
rows = []

inp = sum(t_system + n + OVERHEAD for n in review_tokens) * scale
rows.append(("A. Naive: 1 call per review, verbose JSON", N, inp, VERBOSE_OUT*N, cost(inp, VERBOSE_OUT*N)))
rows.append(("B. + one-word label output", N, inp, LABEL_OUT*N, cost(inp, LABEL_OUT*N)))

trunc = [min(n, TRUNCATE_TO) for n in review_tokens]
inp_c = sum(t_system + n + OVERHEAD for n in trunc) * scale
rows.append((f"C. + truncate reviews to {TRUNCATE_TO} tokens", N, inp_c, LABEL_OUT*N, cost(inp_c, LABEL_OUT*N)))

calls = -(-N // REVIEWS_PER_REQUEST)
inp_d = calls * (t_system + OVERHEAD) + sum(n + 6 for n in trunc) * scale
out_d = N * (LABEL_OUT + 3)
rows.append((f"D. + {REVIEWS_PER_REQUEST} reviews per request", calls, inp_d, out_d, cost(inp_d, out_d)))

cached = (calls - 1) * t_system if t_system >= CACHE_MIN_PREFIX else 0
rows.append(("E. + prompt caching of the prefix", calls, inp_d, out_d, cost(inp_d, out_d, cached)))
rows.append(("F. + async Batch API discount", calls, inp_d, out_d, cost(inp_d, out_d, cached, True)))

dupes = sum(c - 1 for c in Counter(test["text"]).values() if c > 1)
keep = (N - dupes) / N
rows.append((f"G. + response cache ({dupes} duplicate reviews)", int(calls*keep),
             inp_d*keep, out_d*keep, cost(inp_d*keep, out_d*keep, cached*keep, True)))

budget = pd.DataFrame(rows, columns=["strategy", "calls", "input_tokens", "output_tokens", "usd"])
budget["input_tokens"] = budget["input_tokens"].astype(int)
budget["output_tokens"] = budget["output_tokens"].astype(int)
budget["usd"] = budget["usd"].round(2)
print(f"From A to G the bill drops by {1 - budget.usd.iloc[-1]/budget.usd.iloc[0]:.0%}")
budget

**Discuss the trade-offs**

| Strategy | Saves | Costs you |
|---|---|---|
| One-word output | output tokens (the expensive ones) | no explanation for the label |
| Truncation | input tokens | may cut the sentence that decides the sentiment |
| Many reviews per request | repeated instructions | one bad answer can spoil the whole batch |
| Prompt caching | re-reading an identical prefix | keep the variable text **at the end** of the prompt |
| Batch API | ~half the price | results can take hours, so not for live users |
| Response cache | duplicate work | only helps when inputs really repeat |

### A reusable budget guard

In [ ]:
def check_budget(prompt: str, max_input_tokens: int = 4000) -> bool:
    n = len(enc.encode(prompt))
    print(f"Request needs {n} input tokens; budget {max_input_tokens} -> "
          f"{'OK to send' if n <= max_input_tokens else 'REJECT or truncate'}")
    return n <= max_input_tokens

check_budget(SYSTEM + test[0]["text"])
check_budget(SYSTEM + " ".join(test[:20]["text"]))

> **Try it (3 min)**
> 1. Change `PRICE` to the current price of a model your team uses. Which strategy saves the most?
> 2. Set `REVIEWS_PER_REQUEST = 50`. What happens to the cost? What new risk does that create?
> 3. Remove the few-shot examples (`SYSTEM = INSTRUCTIONS`) and re-run. Why does caching stop applying?

---
## Part 6 - Stretch: train your own tokenizer (optional)

Tokenizers are **learned from data**. Train a byte-level BPE tokenizer on 20,000 movie reviews
and compare it with GPT-2's tokenizer, which was trained on general web text.

In [ ]:
from tokenizers import ByteLevelBPETokenizer

corpus = ds["train"].shuffle(seed=3).select(range(20000))["text"]
held_out = ds["test"].shuffle(seed=3).select(range(1000))["text"]

custom = ByteLevelBPETokenizer()
t0 = time.time()
custom.train_from_iterator(corpus, vocab_size=32000, min_frequency=2,
                           special_tokens=["<pad>", "<unk>", "<s>", "</s>"])
print(f"Trained in {time.time()-t0:.1f}s")
os.makedirs("outputs/imdb-bpe", exist_ok=True)
custom.save_model("outputs/imdb-bpe")

longest = sorted(custom.get_vocab(), key=len, reverse=True)[:12]
print("Some learned tokens:", [t.replace("Ġ", "_") for t in longest])

In [ ]:
gpt2 = hf_toks["GPT-2"]
ours = sum(len(custom.encode(t).ids) for t in held_out)
theirs = sum(len(gpt2.encode(t)) for t in held_out)
print(f"1,000 unseen reviews -> ours: {ours:,} tokens | GPT-2: {theirs:,} tokens")

s = "The cinematography was breathtaking but the screenplay felt underwhelming."
print("ours :", show_tokens([t.replace("Ġ", "_") for t in custom.encode(s).tokens]))
print("GPT-2:", show_tokens([t.replace("Ġ", "_") for t in gpt2.tokenize(s)]))

code_text = "def calculate_gst(amount): return amount * 0.18"
print(f"\nOff-domain code -> ours: {len(custom.encode(code_text).ids)} tokens | GPT-2: {len(gpt2.encode(code_text))} tokens")

A domain tokenizer is compact **on its domain** (look at `_cinematography`) and worse outside it.
This is one reason why a model's tokenizer is fixed once the model is trained.

---
## Summary

* `datasets` gives you fast, cached, memory-mapped tables with `map`, `filter`, splits and streaming.
* A tokenizer turns text into **IDs + special tokens + attention mask**. Padding and truncation shape batches.
* The same text costs very different token counts across models and **languages**.
* `max_length` and padding strategy are real speed/accuracy trade-offs.
* Count tokens **before** calling an LLM. Batching, output constraints, prompt caching, batch APIs and response caches can cut cost by more than 90%.